# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [16]:
pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [17]:
# YOUR CHANGES HERE

import numpy as np
import pandas as pd

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = list(dict.fromkeys(rewards["state"]))
actions = list(dict.fromkeys(rewards["action"]))
s_idx = {s: i for i, s in enumerate(states)}
a_idx = {a: i for i, a in enumerate(actions)}

P = np.zeros((len(actions), len(states), len(states)))
for _, r in transitions.iterrows():
    P[a_idx[r["action"]], s_idx[r["state"]], s_idx[r["next_state"]]] = r["probability"]

def reward_matrix(df):
    R = np.zeros((len(actions), len(states)))
    for _, r in df.iterrows():
        R[a_idx[r["action"]], s_idx[r["state"]]] = r["reward"]
    return R

R = reward_matrix(rewards)

def evaluate_policy(policy, R, tol=1e-12, max_iter=100000):
    V = np.zeros(len(states))
    rows = np.arange(len(states))
    for _ in range(max_iter):
        V_new = R[policy, rows] + np.einsum("sj,j->s", P[policy, rows], V)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new
    return V

def value_iteration(R, tol=1e-12, max_iter=100000):
    V = np.zeros(len(states))
    for _ in range(max_iter):
        V_new = (R + P @ V).max(axis=0)
        if np.max(np.abs(V_new - V)) < tol:
            V = V_new
            break
        V = V_new
    Q = R + P @ V
    best = np.array([int(np.flatnonzero(np.isclose(Q[:, s], Q[:, s].max()))[0]) for s in range(len(states))])
    return V, best

do_nothing = np.full(len(states), a_idx["do-nothing"])
V_dn = evaluate_policy(do_nothing, R)
do_nothing_df = pd.DataFrame({"state": states, "expected_discomfort": -V_dn + 0.0})
do_nothing_df

,state,expected_discomfort
0,exposed-1,3.413333
1,exposed-2,4.266667
2,exposed-3,5.333333
3,symptoms-1,6.666667
4,symptoms-2,5.000000
5,symptoms-3,1.666667
6,recovered,0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [18]:
# YOUR CHANGES HERE

do_nothing_df.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [19]:
# YOUR CHANGES HERE

V_opt, pi_opt = value_iteration(R)
min_actions_df = pd.DataFrame({"state": states, "action": [actions[a] for a in pi_opt]})
min_actions_df

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [20]:
# YOUR CHANGES HERE

min_actions_df.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)

Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [21]:
# YOUR CHANGES HERE

V_min = evaluate_policy(pi_opt, R)
min_values_df = pd.DataFrame({"state": states, "expected_discomfort": -V_min + 0.0})
min_values_df

,state,expected_discomfort
0,exposed-1,0.75
1,exposed-2,1.50
2,exposed-3,3.00
3,symptoms-1,6.00
4,symptoms-2,4.50
5,symptoms-3,1.50
6,recovered,0.00


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [22]:
# YOUR CHANGES HERE

min_values_df.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [23]:
# YOUR CHANGES HERE

duration_rewards = rewards[["action", "state"]].copy()
duration_rewards["reward"] = np.where(duration_rewards["state"].str.startswith("symptoms"), -1.0, 0.0)
R_dur = reward_matrix(duration_rewards)
duration_rewards

,action,state,reward
0,do-nothing,exposed-1,0.0
1,do-nothing,exposed-2,0.0
2,do-nothing,exposed-3,0.0
3,do-nothing,symptoms-1,-1.0
4,do-nothing,symptoms-2,-1.0
5,do-nothing,symptoms-3,-1.0
6,do-nothing,recovered,0.0
7,drink-oj,exposed-1,0.0
8,drink-oj,exposed-2,0.0
9,drink-oj,exposed-3,0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [24]:
# YOUR CHANGES HERE

duration_rewards.to_csv("duration-rewards.tsv", sep="\t", index=False)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [25]:
# YOUR CHANGES HERE

V_dur, pi_dur = value_iteration(R_dur)
dur_actions_df = pd.DataFrame({"state": states, "action": [actions[a] for a in pi_dur]})
dur_actions_df

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [26]:
# YOUR CHANGES HERE

dur_actions_df.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [27]:
# YOUR CHANGES HERE

V_days = evaluate_policy(pi_dur, R_dur)
days_df = pd.DataFrame({"state": states, "expected_sick_days": -V_days + 0.0})
days_df

,state,expected_sick_days
0,exposed-1,1.250000
1,exposed-2,2.500000
2,exposed-3,5.000000
3,symptoms-1,10.000000
4,symptoms-2,6.666667
5,symptoms-3,3.333333
6,recovered,0.000000


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [28]:
# YOUR CHANGES HERE

days_df.to_csv("minimum-duration-days.tsv", sep="\t", index=False)

Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [29]:
# YOUR CHANGES HERE

V_speed = evaluate_policy(pi_dur, R)
comparison_df = pd.DataFrame({
    "state": states,
    "speed_discomfort": -V_speed + 0.0,
    "minimize_discomfort": -V_min + 0.0
})
comparison_df

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.833333,0.75
1,exposed-2,1.666667,1.50
2,exposed-3,3.333333,3.00
3,symptoms-1,6.666667,6.00
4,symptoms-2,5.000000,4.50
5,symptoms-3,1.666667,1.50
6,recovered,0.000000,0.00


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [30]:
# YOUR CHANGES HERE

comparison_df.to_csv("policy-comparison.tsv", sep="\t", index=False)

Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.